# Setup

In [1]:
import os, sys

SUMO_HOME  = 'C:\Program Files (x86)\Eclipse\Sumo'
# PROJ_PATH  = '/Library/Frameworks/EclipseSUMO.framework/Versions/1.22.0/EclipseSUMO/framework/EclipseSUMO.framework/Versions/1.22.0/EclipseSUMO/share/proj'

os.environ['SUMO_HOME']  = SUMO_HOME
# os.environ['PROJ_LIB']   = PROJ_PATH
# os.environ['PROJ_DATA']  = PROJ_PATH

sys.path.append(os.path.join(SUMO_HOME, 'tools'))

# Verify all three
checks = {
    'SUMO_HOME':  os.path.exists(SUMO_HOME),
#     'proj.db':    os.path.exists(os.path.join(PROJ_PATH, 'proj.db')),
    'gtfs2pt.py': os.path.exists(f'{SUMO_HOME}/tools/import/gtfs/gtfs2pt.py'),
}
for k, v in checks.items():
    print(f"{k:15s}: {'✅' if v else '❌ NOT FOUND'}")

SUMO_HOME      : ✅
gtfs2pt.py     : ✅


<>:3: SyntaxWarning: invalid escape sequence '\P'
<>:3: SyntaxWarning: invalid escape sequence '\P'
C:\Users\Soheil99\AppData\Local\Temp\ipykernel_14472\516504717.py:3: SyntaxWarning: invalid escape sequence '\P'
  SUMO_HOME  = 'C:\Program Files (x86)\Eclipse\Sumo'


In [2]:
# cd /Users/ziliqu/Dev/SDOT_Worldcup/Sumo_Test/2
!cd "C:\Users\Soheil99\0 codes\DowntownSeattleSUMO\Simulation\GTFS"
!cd

C:\Users\Soheil99\0 codes\DowntownSeattleSUMO\Simulation\GTFS


# OSM Map Extraction

-extraction from Geofrabil for Full Washington Link Network

In [24]:
import requests

# Download Washington state extract from Geofabrik
url = "https://download.geofabrik.de/north-america/us/washington-latest.osm.pbf"
out = "washington.osm.pbf"

print("Downloading Washington state from Geofabrik (~100MB)...")
with requests.get(url, stream=True) as r:
    total = int(r.headers.get('content-length', 0))
    downloaded = 0
    with open(out, 'wb') as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
            downloaded += len(chunk)
            if total:
                pct = downloaded / total * 100
                print(f"\r  {pct:.1f}% ({downloaded/1024/1024:.1f} MB)", end='')

print(f"\n✅ Done — saved to {out}")

  100.0% (339.4 MB)
✅ Done — saved to washington.osm.pbf


In [52]:
!osmium extract \
  -b -122.4500,47.2500,-122.1000,47.8500 \
  washington.osm.pbf \
  -o seattle_link.osm.pbf \
  --strategy complete_ways

No extract specified in config file or on the command line.


In [56]:
!osmium extract \
  -b -124.8,45.5,-116.9,49.1 \
  washington.osm.pbf \
  -o washington_full.osm.pbf \
  --strategy complete_ways

[======================================================================] 100% 


In [36]:
!osmium cat seattle_link.osm.pbf -o seattle_link.osm.xml

[======================================================================] 100% 


In [58]:
!osmium cat washington_full.osm.pbf -o seattle_link_full.osm.xml

[======================================================================] 100% 


# Network Preparation

In [ ]:
!netconvert \
  --osm-files seattle_link.osm.xml \
  -o seattle_lightrail_3.net.xml \
  --type-files $SUMO_HOME/data/typemap/osmNetconvert.typ.xml,$SUMO_HOME/data/typemap/osmNetconvertRailUsage.typ.xml \
  --keep-edges.by-type railway.light_rail,railway.subway \
  --proj.utm true \
  --geometry.remove \
  --junctions.join \
  --output.street-names \
  --ptstop-output seattle_rail_stops.add.xml \
  --ptline-output seattle_rail_ptlines.add.xml \
  --osm.stop-output.length 30

# GTFS

In [3]:
import pandas as pd
import zipfile
import os

def load_gtfs(zip_path):
    data = {}
    with zipfile.ZipFile(zip_path, 'r') as z:
        for file in z.namelist():
            if file.endswith(".txt"):
                data[file] = pd.read_csv(z.open(file))
    return data


### Filtering to major Seattle GTFS Bus Lines

In [9]:

# ==============================
# CONFIG (EDIT THESE)
# ==============================

INPUT_ZIP = "gtfs data/kcm_google_transit.zip"
OUTPUT_ZIP = "gtfs data/kcm_google_transit_downtown.zip"

# Select important downtown routes 
KEEP_ROUTES = [
    "8", "40", "D Line", "E Line"
]
# KEEP_ROUTES = [
#     "1", "2", "3", "4", "7", "8", "40",
#     "101", "120", "C Line", "D Line", "E Line", "H Line"
# ]

# Downtown Seattle bounding box
LAT_MIN, LAT_MAX = 47.58, 47.65
LON_MIN, LON_MAX = -122.37, -122.30

# ==============================
# LOAD GTFS FILES
# ==============================



gtfs = load_gtfs(INPUT_ZIP)

routes = gtfs["routes.txt"]
trips = gtfs["trips.txt"]
stop_times = gtfs["stop_times.txt"]
stops = gtfs["stops.txt"]

# ==============================
# STEP 1: FILTER ROUTES
# ==============================

filtered_routes = routes[routes["route_short_name"].isin(KEEP_ROUTES)]

print(f"Kept routes: {len(filtered_routes)}")

# ==============================
# STEP 2: FILTER TRIPS
# ==============================

filtered_trips = trips[trips["route_id"].isin(filtered_routes["route_id"])]

print(f"Kept trips: {len(filtered_trips)}")

# ==============================
# STEP 3: FILTER STOP TIMES
# ==============================

filtered_stop_times = stop_times[
    stop_times["trip_id"].isin(filtered_trips["trip_id"])
]

# ==============================
# STEP 4: FILTER STOPS (GEOGRAPHIC)
# ==============================

filtered_stops = stops[
    (stops["stop_lat"] >= LAT_MIN) &
    (stops["stop_lat"] <= LAT_MAX) &
    (stops["stop_lon"] >= LON_MIN) &
    (stops["stop_lon"] <= LON_MAX)
]
## TODO ---soheil comment:
# do we have trips where the removed stops are in mid trips? trips that leave simulation area and then come back to it?
print(f"Kept stops in downtown: {len(filtered_stops)}")

# Keep only stop_times that use those stops
filtered_stop_times = filtered_stop_times[
    filtered_stop_times["stop_id"].isin(filtered_stops["stop_id"])
]

# ==============================
# STEP 5: CLEAN TRIPS (remove empty)
# ==============================

valid_trip_ids = filtered_stop_times["trip_id"].unique()
filtered_trips = filtered_trips[filtered_trips["trip_id"].isin(valid_trip_ids)]

# ==============================
# STEP 6: WRITE NEW GTFS ZIP
# ==============================

with zipfile.ZipFile(OUTPUT_ZIP, 'w') as z:
    def write(df, name):
        z.writestr(name, df.to_csv(index=False))

    write(filtered_routes, "routes.txt")
    write(filtered_trips, "trips.txt")
    write(filtered_stop_times, "stop_times.txt")
    write(filtered_stops, "stops.txt")

    # Copy unchanged files if they exist --- soheil: Not sure why not ALL unchanged files are not copied into the new zip
    for fname in ["calendar.txt", "calendar_dates.txt", "agency.txt"]:
        if fname in gtfs:
            write(gtfs[fname], fname)

print(f"✅ Filtered GTFS saved to: {OUTPUT_ZIP}")


Kept routes: 4
Kept trips: 2091
Kept stops in downtown: 649
✅ Filtered GTFS saved to: gtfs data/kcm_google_transit_downtown.zip


In [13]:
# busgtfs = load_gtfs('gtfs data/kcm_google_transit_downtown.zip')
# stops = busgtfs['stops.txt']
# stops.stop_name.unique()
# stops.head()

In [14]:
# railgtfs = load_gtfs('gtfs data/rail_gtfs.zip')
# stops = railgtfs['stops.txt']
# # stops.stop_name.unique()
# # stops.head()
# # stops[stops["stop_name"].str.contains("Lynnwood", case=False, na=False)]
# LAT_MIN, LAT_MAX = 47.5798124250899, 47.65143948560587
# LON_MIN, LON_MAX = -122.38544987028668, -122.29961918065126


# filtered_stops = stops[
#     (stops["stop_lat"] >= LAT_MIN) &
#     (stops["stop_lat"] <= LAT_MAX) &
#     (stops["stop_lon"] >= LON_MIN) &
#     (stops["stop_lon"] <= LON_MAX)
# ]

# filtered_stops

In [6]:
# ==============================
# CONFIG (EDIT THESE)
# ==============================

INPUT_ZIP = "gtfs data/rail_gtfs.zip"
OUTPUT_ZIP = "gtfs data/rail_gtfs_downtown.zip"

# Select important downtown routes 
KEEP_ROUTES = [
    "1 Line", "2 Line"
]

# Downtown Seattle bounding box
# #old ones
LAT_MIN, LAT_MAX = 47.31, 47.84
LON_MIN, LON_MAX = -122.43, -122.09

# new ones
# LAT_MIN, LAT_MAX = 47.5798124250899, 47.65143948560587
# LON_MIN, LON_MAX = -122.38544987028668, -122.29961918065126
# 47.5798124250899, -122.32752854299258 (below SODO)
# 47.65143948560587, -122.30397070065698 (above UW)
# 47.59038693450619, -122.29961918065126 (judkins park)
# 47.59525640734678, -122.38544987028668 (west seattle)

# ==============================
# LOAD GTFS FILES
# ==============================

gtfs = load_gtfs(INPUT_ZIP)

routes = gtfs["routes.txt"]
trips = gtfs["trips.txt"]
stop_times = gtfs["stop_times.txt"]
stops = gtfs["stops.txt"]

# ==============================
# STEP 1: FILTER ROUTES
# ==============================

filtered_routes = routes[routes["route_short_name"].isin(KEEP_ROUTES)]

print(f"Kept routes: {len(filtered_routes)}")

# ==============================
# STEP 2: FILTER TRIPS
# ==============================

filtered_trips = trips[trips["route_id"].isin(filtered_routes["route_id"])]

print(f"Kept trips: {len(filtered_trips)}")

# ==============================
# STEP 3: FILTER STOP TIMES
# ==============================

filtered_stop_times = stop_times[
    stop_times["trip_id"].isin(filtered_trips["trip_id"])
]

# ==============================
# STEP 4: FILTER STOPS (GEOGRAPHIC)
# ==============================

filtered_stops = stops[
    (stops["stop_lat"] >= LAT_MIN) &
    (stops["stop_lat"] <= LAT_MAX) &
    (stops["stop_lon"] >= LON_MIN) &
    (stops["stop_lon"] <= LON_MAX)
]

print(f"Kept stops in downtown: {len(filtered_stops)}")

# Keep only stop_times that use those stops
filtered_stop_times = filtered_stop_times[
    filtered_stop_times["stop_id"].isin(filtered_stops["stop_id"])
]

# ==============================
# STEP 5: CLEAN TRIPS (remove empty)
# ==============================

valid_trip_ids = filtered_stop_times["trip_id"].unique()
filtered_trips = filtered_trips[filtered_trips["trip_id"].isin(valid_trip_ids)]

# ==============================
# STEP 6: WRITE NEW GTFS ZIP
# ==============================

with zipfile.ZipFile(OUTPUT_ZIP, 'w') as z:
    def write(df, name):
        z.writestr(name, df.to_csv(index=False))

    write(filtered_routes, "routes.txt")
    write(filtered_trips, "trips.txt")
    write(filtered_stop_times, "stop_times.txt")
    write(filtered_stops, "stops.txt")

    # Copy unchanged files if they exist
#     for fname in ["calendar.txt", "calendar_dates.txt", "agency.txt"]:
#         if fname in gtfs:
#             write(gtfs[fname], fname)
    for fname in gtfs:
        if fname not in ['routes.txt', 'trips.txt', 'stop_times.txt', 'stops.txt']:
            write(gtfs[fname], fname)

print(f"✅ Filtered GTFS saved to: {OUTPUT_ZIP}")


Kept routes: 2
Kept trips: 17803
Kept stops in downtown: 309
✅ Filtered GTFS saved to: gtfs data/rail_gtfs_downtown.zip


### Converging new Lightrail Segments

In [ ]:
!netconvert -s soheil_seattle.net.xml --remove-edges.by-vclass rail,rail_urban,rail_fast,subway,rail_electric -o soheil_seattle_veh.net.xml

In [ ]:
!netconvert -s seattle_lightrail_2.net.xml,soheil_seattle_veh.net.xml -o merged.net.xml 

#### Adding Full Lightrail Network

In [ ]:
!netconvert -s seattle_lightrail_trimmed.net.xml,soheil_seattle_original.net.xml -o merged.net.xml 

In [ ]:
!netconvert -s seattle_lightrail_trimmed1.net.xml,merged.net.xml  -o merged_final.net.xml 

### Adding GTFS to Sumo Network

In [8]:
# !python $SUMO_HOME/tools/import/gtfs/gtfs2pt.py \
# -n merged.net.xml \
# --gtfs rail_gtfs_fil.zip \
# --date 20260329 \
# --modes=tram \
# --repair \
# --verbose

!python "$SUMO_HOME/tools/import/gtfs/gtfs2pt.py" \
-n merged_v2.net.xml \
--gtfs "gtfs data/rail_gtfs_downtown.zip" \
--date 20260329 \
--modes=tram \
--repair \
--verbose

Loading net
function import_gtfs called at Fri, 19 Jun 2026 19:03:03 +0000
Loading GTFS data "gtfs data/rail_gtfs_downtown.zip"
function import_gtfs finished after 0.687741 seconds
Success.
Success.
Writing fcd file "fcd\gtfs\tram.fcd.xml"
mapping tram
mapping trace with 15 points ... (688 router calls)
mapping trace with 26 points ... (754 router calls)
mapping trace with 22 points ... (716 router calls)
mapping trace with 26 points ... (754 router calls)
mapping trace with 4 points ... (9 router calls)
mapping trace with 8 points ... (21 router calls)
mapping trace with 11 points ... (46 router calls)
mapping trace with 22 points ... (541 router calls)
mapping trace with 11 points ... (46 router calls)
mapping trace with 26 points ... (545 router calls)
mapping trace with 26 points ... (545 router calls)
mapping trace with 1 points ... (0 router calls)
mapping trace with 4 points ... (31 router calls)
mapping trace with 6 points ... (33 router calls)
mapping trace with 10 points ... 

Warning! No mapping library found, falling back to tracemapper.
Warning! No net resources\gtfs\bus.net.xml


In [10]:
!python "$SUMO_HOME/tools/import/gtfs/gtfs2pt.py" \
-n merged_v2.net.xml \
--gtfs "gtfs data/kcm_google_transit_downtown.zip" \
--date 20260617 \
--modes=bus \
--repair \
--verbose

Loading net
function import_gtfs called at Fri, 19 Jun 2026 19:20:05 +0000
Loading GTFS data "gtfs data/kcm_google_transit_downtown.zip"
function import_gtfs finished after 0.063732 seconds
Success.
Writing fcd file "fcd\gtfs\bus.fcd.xml"
Reusing old resources\gtfs\numerical.net.xml
mapping bus
mapping trace with 21 points ...
   Found no candidate edges for 2752.68,30788.71 (index 0)
   Found no candidate edges for 2744.02,31433.51 (index 1)
   Found no candidate edges for 2741.04,31765.47 (index 2)
   Found no candidate edges for 2768.98,31977.29 (index 3)
   Found no candidate edges for 2865.49,34119.74 (index 4)
   Found no candidate edges for 2580.02,34051.45 (index 5)
   Found no candidate edges for 2339.52,34080.25 (index 6)
7 Points had no candidates. (21821 router calls)
mapping trace with 23 points ...
   Found no candidate edges for 2315.01,34065.19 (index 15)
   Found no candidate edges for 2641.47,34037.16 (index 16)
   Found no candidate edges for 2849.22,34082.70 (index 

Warning! No mapping library found, falling back to tracemapper.
Trip 800890320 (bus): detour (factor 26.79) to stop index 20, fromPos=-1278.53,34334.40 toPos=-1373.33,34503.20 (airLine=193.60 path=5186.93)
Trip 800890360.trimmed (bus): detour (factor 26.79) to stop index 2, fromPos=-1278.53,34334.40 toPos=-1373.33,34503.20 (airLine=193.60 path=5186.93)
Trip 800891710.trimmed (bus): detour (factor 10.69) to stop index 2, fromPos=-315.11,33826.49 toPos=9.97,33827.98 (airLine=325.08 path=3475.54)
Trip 837914310 (bus): detour (factor 13.63) to stop index 9, fromPos=-3.53,34203.53 toPos=-9.85,34609.64 (airLine=406.17 path=5534.80)
Trip 585675840 (bus): detour (factor 26.79) to stop index 9, fromPos=-1278.53,34334.40 toPos=-1373.33,34503.20 (airLine=193.60 path=5186.93)
Trip 732748430 (bus): detour (factor 17.55) to stop index 1, fromPos=-1693.07,34487.42 toPos=-1392.33,34437.75 (airLine=304.82 path=5350.14)
Trip 761138480 (bus): detour (factor 17.55) to stop index 1, fromPos=-1693.07,34487.

In [ ]:

with open("tram.add.xml", "w") as f:
    f.write("""<additional>
    <vType id="tram"
           vClass="rail"
           accel="1.2"
           decel="1.5"
           sigma="0.3"
           length="40"
           maxSpeed="25"
           guiShape="rail"/>
</additional>
""")


In [6]:
with open("bus.add.xml", "w") as f:
    f.write("""<additional>
    <vType id="bus"
           vClass="bus"
           accel="1.0"
           decel="4.5"
           sigma="0.5"
           length="12"
           maxSpeed="16.7"
           guiShape="bus"/>
</additional>
""")


In [12]:
import re

with open("gtfs_pt_stops.add.xml") as f:
    content = f.read()

# replace trainStop with busStop
content = content.replace("trainStop", "busStop")

with open("gtfs_pt_stops_fixed.add.xml", "w") as f:
    f.write(content)


In [ ]:
!sumo-gui \
-n merged.net.xml \
-a tram.add.xml,gtfs_pt_stops_fixed.add.xml,gtfs_pt_vehicles.add.xml


In [ ]:
!sumo-gui \
-n merged.net.xml \
-a bus.add.xml,gtfs_pt_stops.add.xml,gtfs_pt_vehicles.add.xml